# π0.5 q/k/v/o checkpoint実動作選抜（再学習なし）

Driveに保存済みの1500・2000・2500・3000-step LoRA checkpointを、元の公開π0.5へ一つずつ統合し、同一の公開4タスク・初期状態・policy乱数で比較します。現在の0.369提出ZIPは読み書きせず、候補ごとの評価JSON・ログ・失敗動画と比較summaryだけをDriveへ保存します。


In [ ]:
from pathlib import Path
import json
import os
import subprocess

REPO_URL = "https://github.com/KosukeKomeya/PARC2026_pre.git"
BRANCH = "feature/experiment"
REPO_DIR = Path("/content/PARC2026_pre")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR, check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"既存パスがGitリポジトリではありません: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip())
%cd /content/PARC2026_pre


## 元モデルと固定ランタイムを再構築

新しいランタイムなので、学習は行わず、元の公開π0.5・PaliGemma tokenizer・固定LeRobot v0.4.4だけを再取得します。PaliGemmaの利用条件へ同意済みのHugging Face read tokenを使用します。


In [ ]:
%pip install -q "huggingface-hub>=0.34.2,<0.36.0"
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
PI05_PYTHON = Path("/content/pi05_py310/bin/python")
if PI05_PYTHON.is_file():
    setup_command = ["python", "examples/pi05_parc_colab_setup.py", "--reuse-runtime", "--skip-download"]
else:
    setup_command = ["python", "examples/pi05_parc_colab_setup.py"]
subprocess.run(setup_command, cwd=REPO_DIR, check=True)
BASE_MODEL_DIR = REPO_DIR / "submission_template/model_weights/pi05_libero_finetuned_v044"
if (BASE_MODEL_DIR / "pi05_lora_merge_manifest.json").exists():
    raise RuntimeError("元モデルがLoRA統合版に置換されています。新しいランタイムで再実行してください。")
for required in ["config.json", "model.safetensors", "policy_preprocessor.json", "policy_postprocessor.json"]:
    assert (BASE_MODEL_DIR / required).is_file(), required
print("ORIGINAL_BASE_READY", BASE_MODEL_DIR)


## Drive checkpointを検出

過去の操作で`batch4`と`batcho4`の両方が作られた可能性があるため、必要な4 checkpointが最も多く揃っている方を自動選択します。新しい評価結果は綴りを直した`batch4/checkpoint_sweep`へ保存します。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CANDIDATE_STEPS = [1500, 2000, 2500, 3000]
drive_candidates = [
    Path("/content/drive/MyDrive/PARC2026/pi05_action_expert_lora_qkvo_batch4"),
    Path("/content/drive/MyDrive/PARC2026/pi05_action_expert_lora_qkvo_batcho4"),
]

def complete_steps(root):
    found = []
    for step in CANDIDATE_STEPS:
        checkpoint = root / "checkpoints" / f"{step:06d}"
        required = [
            checkpoint / "pretrained_model/adapter_config.json",
            checkpoint / "pretrained_model/adapter_model.safetensors",
            checkpoint / "training_state/training_step.json",
        ]
        if all(path.is_file() for path in required):
            found.append(step)
    return found

inventory = [(root, complete_steps(root)) for root in drive_candidates]
for root, steps in inventory:
    print(root, "steps=", steps)
SOURCE_DRIVE_ROOT, available_steps = max(inventory, key=lambda item: len(item[1]))
if available_steps != CANDIDATE_STEPS:
    raise RuntimeError(f"必要checkpointが揃っていません: root={SOURCE_DRIVE_ROOT}, steps={available_steps}")
OUTPUT_DRIVE_ROOT = drive_candidates[0]
OUTPUT_DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = SOURCE_DRIVE_ROOT / "checkpoints"
manifest_candidates = [
    SOURCE_DRIVE_ROOT / "data_split.json",
    Path("/content/drive/MyDrive/PARC2026/pi05_action_expert_lora_full40/data_split.json"),
]
TRAINING_MANIFEST = next((path for path in manifest_candidates if path.is_file()), None)
if TRAINING_MANIFEST is None:
    raise FileNotFoundError("data_split.jsonがDriveにありません")
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("OUTPUT_DRIVE_ROOT:", OUTPUT_DRIVE_ROOT)
print("TRAINING_MANIFEST:", TRAINING_MANIFEST)


## 公開4タスク評価環境を準備


In [ ]:
system_packages = [
    "libosmesa6", "libgl1", "libglfw3", "libglew2.2",
    "libegl1", "libsm6", "libxext6", "libxrender1",
    "libglib2.0-0", "libmagickwand-dev", "unzip",
]
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "--no-install-recommends", *system_packages], check=True)
setup_env = os.environ.copy()
setup_env.update({"PYTHON": str(PI05_PYTHON), "MUJOCO_GL": "egl", "MPLBACKEND": "Agg"})
subprocess.run(["bash", "setup.sh"], cwd=REPO_DIR, env=setup_env, check=True)
EVAL_PYTHON = REPO_DIR / "venv/bin/python"
if not EVAL_PYTHON.is_file():
    raise FileNotFoundError(EVAL_PYTHON)
print("EVALUATION_RUNTIME_READY", PI05_PYTHON, EVAL_PYTHON)


## 4 checkpointを同一条件で比較

各候補を3 episodes/task（12 episodes）で評価します。候補ごとにDriveへ結果を保存するため、ランタイムが切れても完了済み候補は再実行しません。policy乱数もタスク・episodeごとに固定し、checkpoint間の比較ノイズを減らします。最終ZIPは作成・上書きしません。


In [ ]:
sweep_command = [
    "python", "-u", "examples/pi05_checkpoint_sweep.py",
    "--repo-root", str(REPO_DIR),
    "--policy-python", str(PI05_PYTHON),
    "--eval-python", str(EVAL_PYTHON),
    "--checkpoint-root", str(CHECKPOINT_ROOT),
    "--output-drive-root", str(OUTPUT_DRIVE_ROOT),
    "--training-manifest", str(TRAINING_MANIFEST),
    "--steps", "1500", "2000", "2500", "3000",
    "--baseline-step", "2000",
    "--episodes", "3",
    "--max-steps", "600",
    "--seed", "42",
    "--policy-seed", "20260814",
    "--replan-steps", "10",
    "--inference-steps", "10",
    "--record-video",
]
process = subprocess.Popen(
    sweep_command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"checkpoint sweep failed with exit={returncode}")


## 比較結果を確認

推奨候補は成功率を最優先、次に衝突率、同率ならsteps・軌道距離・回転・jerk・SPARCで2000-step基準より改善した指標数を使います。非公開の採点正規化は推測していません。この段階では推奨候補を提出せず、上位候補を5 episodes/taskで確認します。


In [ ]:
summary_path = OUTPUT_DRIVE_ROOT / "checkpoint_sweep/checkpoint_sweep_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("recommended for confirmation:", summary["recommended_step_for_confirmation"])
print("summary:", summary_path)
print("現在の最終提出ZIPは変更していません。")
